# Laboratorio · Cuando y no es un número cualquiera

**La pregunta:** ¿esta tienda va a pedir tequila la próxima semana? Y si pide, ¿cuántas botellas?

Usamos las ventas reales de licor del estado de Iowa (Iowa Liquor Sales, dominio público). Cada fila es una tienda en una semana en que compró algo al estado, y la variable a explicar es cuántas botellas pidió de un solo producto: Jose Cuervo Especial Reposado de 750 ml. Son 400 tiendas, de marzo de 2012 a agosto de 2016.

Es un problema de demanda intermitente, como el de refacciones: la mayoría de las semanas no se pide nada y cuando se pide llega una caja completa. En la segunda parte volvemos al vodka de la clase pasada para modelar participaciones de mercado.

Las variables explicativas usan solo semanas anteriores (no hay fuga), y evaluamos siempre en el último año, sin tocarlo para entrenar. Antes de cada celda marcada, escribe tu predicción.

| Columna | Qué es |
| --- | --- |
| `pidio`, `botellas` | Si la tienda pidió el tequila esa semana y cuántas botellas |
| `tam_tienda` | Log de las botellas semanales de todos los productos en las 8 semanas anteriores |
| `pedidos_ult8`, `botellas_ult8` | Semanas con pedido y botellas pedidas del tequila en las 8 semanas activas anteriores |
| `semanas_desde_ultimo` | Semanas desde el último pedido del tequila (tope 52) |
| `feriado` | Semana de Acción de Gracias, Navidad o Año Nuevo |

## Preparación

In [ ]:
# Funciona igual en tu computadora y en Colab (en Colab descarga el repo la primera vez)
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/glm-clasificacion-lab"):
        subprocess.run(["git", "clone", "-q", "https://github.com/nielspac177/glm-clasificacion-lab.git", "/content/glm-clasificacion-lab"], check=True)
    os.chdir("/content/glm-clasificacion-lab/notebooks")
sys.path.append("../src")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de cambios de nombre en scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, recall_score, precision_score

from glm_lab import (cargar_tequila, cargar_vodka, separar_por_fecha, momios, inv_logit, umbral_por_costos,
                     costo_total, brier, tabla_calibracion, dispersion_pearson, prob_cero_poisson, prob_cero_nb,
                     cuantil_nb)

d = cargar_tequila("../data/iowa_tequila_tienda_semana.csv")
d["log_bot8"] = np.log1p(d["botellas_ult8"])          # botellas pedidas en las 8 semanas anteriores, en log
d["log_desde"] = np.log1p(d["semanas_desde_ultimo"])  # semanas desde el último pedido, en log
entrena, prueba = separar_por_fecha(d, "2015-09-01")
X = "tam_tienda + pedidos_ult8 + log_bot8 + log_desde + feriado + C(mes)"
print(f"entrenamiento: {len(entrena):,} filas · prueba (último año): {len(prueba):,} filas")
d.head()

## Nivel 0 · Mirar antes de modelar

**Predice antes de correr.** ¿Qué porcentaje de las tienda-semanas crees que tiene cero botellas? ¿Y cuál será el valor más común cuando sí hay pedido?

_Tu predicción:_ 

In [ ]:
y = d["botellas"]
print(f"semanas sin pedido: {(y == 0).mean():.1%}   media: {y.mean():.2f}   varianza: {y.var():.1f}")
y[y > 0].value_counts(normalize=True).head(8).round(3)

**Explica.** ¿Qué te dice que la varianza sea 26 veces la media? ¿Y el pico en 12?

_Tu explicación:_ 

## Nivel 1 · ¿Qué pasa si usamos la regresión de la clase pasada?

Ajustamos mínimos cuadrados a la variable 0/1 (el *modelo de probabilidad lineal*) y a las botellas.

**Predice antes de correr.** ¿Qué fracción de las predicciones de probabilidad caerá fuera de [0, 1] en el último año? ¿Y cuántas predicciones de botellas serán negativas?

_Tu predicción:_ 

In [ ]:
mpl = smf.ols("pidio ~ " + X, entrena).fit()
p_mpl = mpl.predict(prueba)
mco_bot = smf.ols("botellas ~ " + X, entrena).fit()
b_mco = mco_bot.predict(prueba)
print(f"probabilidades fuera de [0, 1]: {((p_mpl < 0) | (p_mpl > 1)).mean():.1%}  (mín {p_mpl.min():.3f}, máx {p_mpl.max():.3f})")
print(f"botellas negativas: {(b_mco < 0).mean():.1%}  (mínimo {b_mco.min():.2f})")

**Explica.** ¿La recta falla siempre?

_Tu explicación:_ 

## Nivel 2 · Regresión logística

El modelo es lineal en los **log-momios**: log(p / (1 − p)) = a + b·x. Se estima por máxima verosimilitud.

**Ejercicio.** Ajusta la logística con la misma fórmula `X` usando `smf.logit` y guarda el resultado en `logit`.

In [ ]:
logit = smf.___("pidio ~ " + X, entrena).fit(disp=0)
logit.params.drop([c for c in logit.params.index if "mes" in c]).round(3)

Un coeficiente de la logística se lee en log-momios. Para llevarlo a algo legible hay dos caminos: elevar e (razón de momios) o calcular el cambio en probabilidad.

In [ ]:
razon = np.exp(logit.params[["tam_tienda", "pedidos_ult8", "feriado"]])
ame = logit.get_margeff().summary_frame().loc[["tam_tienda", "pedidos_ult8", "feriado"], "dy/dx"]
pd.DataFrame({"coef (log-momios)": logit.params[razon.index], "razón de momios": razon,
              "coef / 4": logit.params[razon.index] / 4, "efecto promedio en p": ame}).round(3)

**Explica.** Una semana más con pedido en las últimas 8: ¿cuánto cambia la probabilidad de pedir?

_Tu explicación:_ 

## Nivel 3 · Lo que scikit-learn hace sin avisar

`LogisticRegression()` de scikit-learn **penaliza por defecto** (Ridge con `C=1`). Tomamos una muestra chica de 200 filas.

**Predice antes de correr.** Con 200 filas, ¿se parecerán mucho los coeficientes con y sin penalización?

_Tu predicción:_ 

In [ ]:
cols = ["tam_tienda", "pedidos_ult8", "log_bot8", "log_desde", "feriado"]
muestra = entrena.sample(200, random_state=86)
por_defecto = LogisticRegression(max_iter=5000).fit(muestra[cols], muestra["pidio"]).coef_[0]
sin_penalizar = LogisticRegression(penalty=None, max_iter=5000).fit(muestra[cols], muestra["pidio"]).coef_[0]
print(pd.DataFrame({"C=1 (por defecto)": por_defecto, "sin penalizar": sin_penalizar}, index=cols).round(2))
pd.crosstab(muestra["feriado"], muestra["pidio"])

**Explica.** ¿Qué pasó con `feriado`?

_Tu explicación:_ 

## Nivel 4 · LDA: la otra forma de clasificar

La logística modela directamente P(pide | x). El análisis discriminante lineal (LDA) modela cómo se distribuye x **dentro** de cada clase (normal, con la misma covarianza) y luego usa Bayes para obtener P(pide | x). La frontera también es lineal.

**Ejercicio.** Ajusta `LinearDiscriminantAnalysis()` con las columnas `cols` y calcula su AUC en prueba, igual que la logística.

In [ ]:
modelos = {"logística": LogisticRegression(penalty=None, max_iter=5000), "LDA": ___,
           "QDA": QuadraticDiscriminantAnalysis()}
for nombre, m in modelos.items():
    q = m.fit(entrena[cols], entrena["pidio"]).predict_proba(prueba[cols])[:, 1]
    print(f"{nombre:10s} AUC {roc_auc_score(prueba['pidio'], q):.3f}   log-loss {log_loss(prueba['pidio'], q):.4f}")

Con 60 mil filas, la logística y LDA quedan prácticamente empatadas. QDA ordena mejor a las tiendas (AUC más alto) pero sus probabilidades son peores (log-loss más alto): ordenar bien y dar probabilidades creíbles son dos cosas distintas. ¿Y con pocos datos? Repetimos 200 veces el ajuste con muestras chicas y medimos el log-loss en el último año (más bajo es mejor).

**Predice antes de correr.** Con 30 o 60 filas de entrenamiento, ¿cuál crees que gana: LDA o la logística?

_Tu predicción:_ 

In [ ]:
rng = np.random.default_rng(0)
filas = []
for n in [30, 60, 120, 250, 1000]:
    for k in range(200):
        m = entrena.iloc[rng.choice(len(entrena), n, replace=False)]
        if m["pidio"].sum() < 2:
            continue
        for nombre, mod in [("logística", LogisticRegression(penalty=None, max_iter=5000)), ("LDA", LinearDiscriminantAnalysis())]:
            q = np.clip(mod.fit(m[cols], m["pidio"]).predict_proba(prueba[cols])[:, 1], 1e-6, 1 - 1e-6)
            filas.append({"n": n, "modelo": nombre, "log_loss": log_loss(prueba["pidio"], q)})
pd.DataFrame(filas).pivot_table(index="n", columns="modelo", values="log_loss", aggfunc="median").round(3)

**Explica.** ¿Por qué LDA gana con pocos datos y empata con muchos?

_Tu explicación:_ 

## Nivel 5 · ¿Qué tan bueno es el clasificador?

**Predice antes de correr.** Un modelo que dice siempre 'no pide', ¿qué exactitud tendrá en el último año? ¿Y la logística con umbral 0.5?

_Tu predicción:_ 

In [ ]:
y_te = prueba["pidio"].to_numpy()
p_log = logit.predict(prueba).to_numpy()
print(f"exactitud de 'nunca pide': {1 - y_te.mean():.1%}")
print(f"logística con umbral 0.5: exactitud {np.mean((p_log >= 0.5) == y_te):.1%}, recall {recall_score(y_te, p_log >= 0.5):.1%}")
print(f"AUC ROC {roc_auc_score(y_te, p_log):.3f}   precisión promedio (área PR) {average_precision_score(y_te, p_log):.3f}"
      f"   tasa de positivos {y_te.mean():.3f}")

**Explica.** ¿Por qué la exactitud no sirve aquí y qué miramos en su lugar?

_Tu explicación:_ 

**¿Las probabilidades son creíbles?** Una tabla de calibración agrupa semanas con probabilidad parecida y compara con lo que pasó.

In [ ]:
cal = tabla_calibracion(y_te, p_log, n_grupos=10)
print(f"Brier del modelo {brier(y_te, p_log):.4f}   Brier de predecir la tasa {brier(y_te, np.full_like(p_log, entrena['pidio'].mean())):.4f}")
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 0.8], [0, 0.8], "--", color="gray")
ax.scatter(cal["p_media"], cal["frecuencia"], s=np.sqrt(cal["n"]) * 3)
ax.set_xlabel("probabilidad predicha"); ax.set_ylabel("frecuencia observada")
plt.show()
cal.round(3)

**Del modelo a la decisión.** Supón que mandar al vendedor a una tienda que no iba a pedir cuesta 1 y no ir a una que sí iba a pedir cuesta 5. Con probabilidades calibradas, el umbral que minimiza el costo es costo_FP / (costo_FP + costo_FN) (Elkan, 2001).

**Ejercicio.** Calcula el umbral con `umbral_por_costos` y compara el costo total contra el umbral 0.5 con `costo_total`.

In [ ]:
u = umbral_por_costos(costo_fp=___, costo_fn=___)
for umbral in [0.5, u]:
    print(f"umbral {umbral:.3f}: costo {costo_total(y_te, p_log, umbral, 1, 5):,}   "
          f"recall {recall_score(y_te, p_log >= umbral):.0%}   precisión {precision_score(y_te, p_log >= umbral):.0%}")
print(f"nunca ir: {costo_total(y_te, np.zeros_like(p_log), 0.5, 1, 5):,}")

**Explica.** ¿Por qué el umbral no es 0.5?

_Tu explicación:_ 

## Nivel 6 · Contar botellas: Poisson y binomial negativa

La regresión de Poisson usa enlace log: log E[y] = a + b·x. Un coeficiente se lee como razón de tasas: e^b es por cuánto se multiplica la media.

**Predice antes de correr.** La Poisson ajustada, ¿qué porcentaje de semanas sin pedido predecirá en el último año? (Observado: 83%)

_Tu predicción:_ 

In [ ]:
poisson = smf.glm("botellas ~ " + X, entrena, family=sm.families.Poisson()).fit()
mu_p = poisson.predict(prueba)
print(f"dispersión de Pearson: {dispersion_pearson(entrena['botellas'], poisson.fittedvalues, len(poisson.params)):.1f}  (Poisson supone 1)")
print(f"ceros: observados {(prueba['botellas'] == 0).mean():.1%}   Poisson predice {prob_cero_poisson(mu_p).mean():.1%}")

**Ejercicio.** Ajusta la binomial negativa con `smf.negativebinomial` y la misma fórmula; guarda `alpha` y la media predicha `mu_nb`.

In [ ]:
bn = smf.___("botellas ~ " + X, entrena).fit(disp=0, maxiter=200)
alpha, mu_nb = bn.params["alpha"], bn.predict(prueba)
print(f"alpha = {alpha:.1f}   ceros predichos {prob_cero_nb(mu_nb, alpha).mean():.1%}   AIC Poisson {poisson.aic:,.0f}   AIC BN {bn.aic:,.0f}")

In [ ]:
robusta = smf.glm("botellas ~ " + X, entrena, family=sm.families.Poisson()).fit(cov_type="HC1")
pd.DataFrame({"razón de tasas": np.exp(poisson.params[["pedidos_ult8", "feriado"]]),
              "EE Poisson": poisson.bse[["pedidos_ult8", "feriado"]],
              "EE Poisson robusto": robusta.bse[["pedidos_ult8", "feriado"]],
              "EE binomial negativa": bn.bse[["pedidos_ult8", "feriado"]]}).round(4)

**Explica.** ¿Qué estaba mal con la Poisson y qué no?

_Tu explicación:_ 

## Nivel 7 · Dos preguntas separadas: ¿pide? y ¿cuánto?

Un modelo *hurdle* (Mullahy, 1986) multiplica dos modelos: la probabilidad de pedir (la logística del nivel 2) y el tamaño del pedido cuando sí hay pedido. Es la misma idea de Croston (1972) para demanda intermitente.

In [ ]:
con_pedido = entrena[entrena["botellas"] > 0]
tamano = smf.glm("botellas ~ " + X, con_pedido, family=sm.families.Poisson()).fit()
hurdle = p_log * tamano.predict(prueba)
yb = prueba["botellas"].to_numpy()
pronosticos = {"cero siempre": np.zeros_like(yb, dtype=float), "promedio 8 semanas": prueba["botellas_ult8"] / 8,
               "MCO": b_mco, "Poisson": mu_p, "binomial negativa": mu_nb, "hurdle": hurdle}
pd.DataFrame({n: {"MAE": np.mean(np.abs(yb - f)), "RMSE": np.sqrt(np.mean((yb - f) ** 2))}
              for n, f in pronosticos.items()}).T.round(3)

**Explica.** ¿Por qué 'cero siempre' tiene el mejor MAE? ¿Significa que es el mejor pronóstico?

_Tu explicación:_ 

## Nivel 8 · De la distribución a un máximo de inventario

Un máximo con nivel de servicio de 95% es el cuantil 0.95 de la demanda de la semana. Lo calculamos con tres supuestos y medimos la **cobertura**: en qué fracción de semanas del último año la demanda quedó por debajo del máximo.

**Predice antes de correr.** ¿Cuál crees que se acerca más al 95% prometido: la normal (regresión lineal), la Poisson o la binomial negativa?

_Tu predicción:_ 

In [ ]:
filas = []
for q in [0.90, 0.95]:
    maximos = {"normal (MCO)": b_mco + stats.norm.ppf(q) * np.sqrt(mco_bot.scale),
               "Poisson": stats.poisson(mu_p).ppf(q),
               "binomial negativa": cuantil_nb(mu_nb, alpha, q)}
    for nombre, m in maximos.items():
        filas.append({"nivel": q, "supuesto": nombre, "cobertura": np.mean(yb <= m), "máximo promedio": np.mean(m)})
pd.DataFrame(filas).round(3)

**Explica.** ¿Qué supuesto usarías para fijar un máximo y por qué?

_Tu explicación:_ 

## Nivel 9 · Participaciones de mercado: un número entre 0 y 1

Volvemos al vodka de la clase pasada. La participación (`share`) de cada proveedor vive entre 0 y 1. El *logit fraccional* (Papke y Wooldridge, 1996) usa la verosimilitud binomial con enlace logit sobre la proporción, con errores robustos.

In [ ]:
v = cargar_vodka("../data/iowa_vodka_semanal.csv")
v["log_precio"] = np.log(v["precio_litro"])
mco_share = smf.ols("share ~ log_precio + C(proveedor)", v).fit()
frac = smf.glm("share ~ log_precio + C(proveedor)", v, family=sm.families.Binomial()).fit(cov_type="HC1")
print(f"MCO: {mco_share.params['log_precio']:.3f} puntos de share por unidad de log precio")
print(f"logit fraccional: {frac.params['log_precio']:.2f} en log-momios (EE robusto {frac.bse['log_precio']:.2f})")

**Predice antes de correr.** Si cada proveedor sube 50% su precio, ¿alguno tendrá share negativo según MCO?

_Tu predicción:_ 

In [ ]:
filas = []
for prov, g in v.groupby("proveedor"):
    g = g.assign(log_precio=g["log_precio"] + np.log(1.5))
    filas.append({"proveedor": prov, "share actual": v.loc[v["proveedor"] == prov, "share"].mean(),
                  "MCO": mco_share.predict(g).mean(), "semanas negativas (MCO)": (mco_share.predict(g) < 0).mean(),
                  "logit fraccional": frac.predict(g).mean()})
pd.DataFrame(filas).round(3)

**Explica.** ¿Qué ganas con el logit fraccional en tu proyecto de share?

_Tu explicación:_ 

## Para tu proyecto

1. ¿Tu y es continua, 0/1, conteo o proporción? ¿Qué distribución y qué enlace le corresponden?
2. Si es un clasificador, ¿cuánto cuesta cada tipo de error? ¿De ahí sale tu umbral?
3. ¿Tus probabilidades están calibradas en el rango donde vas a decidir?
4. Si son conteos, ¿la varianza es mucho mayor que la media? ¿Hay más ceros de los que predice tu modelo?
5. ¿Evalúas la distribución o solo un promedio? ¿Con qué métrica?